# DiD: Minimum Wage Effects on Employment

**Module 04 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Apply the difference-in-differences estimator to a labour economics dataset
2. Implement the 2×2 DiD manually and via regression
3. Use CausalPy's `DifferenceInDifferences` class for Bayesian DiD
4. Interpret the DiD estimate as an ATT with appropriate uncertainty

## Background

Card and Krueger (1994) is the most cited DiD study in economics. They examined whether New Jersey's 1992 minimum wage increase from $4.25 to $5.05 per hour affected fast food employment. Their design:

- **Treated group:** New Jersey fast food restaurants
- **Control group:** Pennsylvania fast food restaurants (no minimum wage change)
- **Pre-period:** February 1992 (before NJ increase)
- **Post-period:** November 1992 (after NJ increase)

We'll replicate the design with synthetic data that reproduces the key features of the original.

In [ ]:
learning_objectives(["Apply the difference-in-differences estimator to a labour economics dataset", "Implement the 2×2 DiD manually and via regression", "Use CausalPy's `DifferenceInDifferences` class for Bayesian DiD", "Interpret the DiD estimate as an ATT with appropriate uncertainty"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
import causalpy as cp
import arviz as az
import warnings
warnings.filterwarnings('ignore')

np.random.seed(1994)  # Card & Krueger year

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Simulate the Card & Krueger Dataset

The original dataset surveyed ~410 New Jersey and ~357 Pennsylvania fast food restaurants before and after NJ's minimum wage increase. Key facts from the original study:

- NJ pre-period mean FTE: 20.4 employees
- PA pre-period mean FTE: 23.3 employees  
- NJ post-period mean FTE: 21.0 employees
- PA post-period mean FTE: 21.2 employees

The raw DiD estimate: (21.0 - 20.4) - (21.2 - 23.3) = 0.6 - (-2.1) = **2.76 FTE**

Contrary to the prediction that minimum wage increases destroy jobs, employment actually rose in NJ relative to PA.

In [ ]:
# Reproduce Card & Krueger (1994) data structure
# Parameters from the original Table 2
NJ_PRE_MEAN = 20.4
NJ_POST_MEAN = 21.03
PA_PRE_MEAN = 23.33
PA_POST_MEAN = 21.17
TRUE_DiD = (NJ_POST_MEAN - NJ_PRE_MEAN) - (PA_POST_MEAN - PA_PRE_MEAN)

n_nj = 410
n_pa = 357
sigma = 8.0  # standard deviation of FTE employment

# Generate NJ restaurants (treated)
nj_pre = np.random.normal(NJ_PRE_MEAN, sigma, n_nj // 2)
nj_post = np.random.normal(NJ_POST_MEAN, sigma, n_nj // 2)

# Generate PA restaurants (control)
pa_pre = np.random.normal(PA_PRE_MEAN, sigma, n_pa // 2)
pa_post = np.random.normal(PA_POST_MEAN, sigma, n_pa // 2)

# Combine into long-format panel
df = pd.DataFrame({
    'fte': np.concatenate([nj_pre, nj_post, pa_pre, pa_post]),
    'state': (['NJ'] * len(nj_pre) + ['NJ'] * len(nj_post) +
              ['PA'] * len(pa_pre) + ['PA'] * len(pa_post)),
    'period': (['pre'] * len(nj_pre) + ['post'] * len(nj_post) +
               ['pre'] * len(pa_pre) + ['post'] * len(pa_post)),
    'chain': np.random.choice(['Burger King', 'KFC', 'Wendys', 'Roy Rogers'],
                               n_nj + n_pa)
})

# Create binary indicators for CausalPy / regression
df['treated'] = (df['state'] == 'NJ').astype(int)
df['post'] = (df['period'] == 'post').astype(int)

# Winsorise at 0 (FTE can't be negative)
df['fte'] = df['fte'].clip(lower=0)

print(f"Dataset shape: {df.shape}")
print(f"True DiD (Card & Krueger): {TRUE_DiD:.2f} FTE")
print(f"\nGroup means:")
print(df.groupby(['state', 'period'])['fte'].mean().round(2))

## 2. Manual DiD Calculation

The DiD estimator is:

$$\hat{\tau}_{DiD} = (\bar{Y}_{NJ,post} - \bar{Y}_{NJ,pre}) - (\bar{Y}_{PA,post} - \bar{Y}_{PA,pre})$$

Let's compute this directly from the group means.

In [ ]:
# Compute group means
means = df.groupby(['state', 'period'])['fte'].mean().unstack()

nj_pre_mean = means.loc['NJ', 'pre']
nj_post_mean = means.loc['NJ', 'post']
pa_pre_mean = means.loc['PA', 'pre']
pa_post_mean = means.loc['PA', 'post']

# DiD components
delta_nj = nj_post_mean - nj_pre_mean   # NJ change
delta_pa = pa_post_mean - pa_pre_mean   # PA change (counterfactual)
did_estimate = delta_nj - delta_pa      # DiD

print("=" * 45)
print("MANUAL DiD CALCULATION")
print("=" * 45)
print(f"NJ (Treated)")
print(f"  Pre-period:  {nj_pre_mean:.2f} FTE")
print(f"  Post-period: {nj_post_mean:.2f} FTE")
print(f"  Change (ΔNJ): {delta_nj:+.2f} FTE")
print()
print(f"PA (Control)")
print(f"  Pre-period:  {pa_pre_mean:.2f} FTE")
print(f"  Post-period: {pa_post_mean:.2f} FTE")
print(f"  Change (ΔPA): {delta_pa:+.2f} FTE")
print()
print(f"DiD Estimate (ΔNJ - ΔPA): {did_estimate:+.2f} FTE")
print(f"True value (Card & Krueger): {TRUE_DiD:+.2f} FTE")

## 3. Visualising the DiD

The standard DiD visualisation shows the two group means over time, with the counterfactual trajectory (dashed line) showing what NJ would have looked like absent the minimum wage increase.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: DiD visualisation ---
ax = axes[0]
period_numeric = {'pre': 0, 'post': 1}

# Plot observed means
ax.plot([0, 1], [nj_pre_mean, nj_post_mean],
        'o-', color='steelblue', linewidth=2.5, markersize=8, label='NJ (Treated)')
ax.plot([0, 1], [pa_pre_mean, pa_post_mean],
        's-', color='darkorange', linewidth=2.5, markersize=8, label='PA (Control)')

# Counterfactual: NJ following PA's trend
nj_counterfactual = nj_pre_mean + delta_pa
ax.plot([0, 1], [nj_pre_mean, nj_counterfactual],
        '--', color='steelblue', linewidth=2, alpha=0.6, label='NJ Counterfactual')

# Annotate treatment effect
ax.annotate('', xy=(1.05, nj_post_mean), xytext=(1.05, nj_counterfactual),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(1.08, (nj_post_mean + nj_counterfactual) / 2,
        f'DiD = {did_estimate:+.2f}\nFTE', color='red', fontsize=10, va='center')

ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
ax.text(0.52, 19, 'NJ min wage\nincrease', color='gray', fontsize=9)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre (Feb 1992)', 'Post (Nov 1992)'])
ax.set_ylabel('Mean FTE Employment')
ax.set_title('Difference-in-Differences: Card & Krueger (1994)')
ax.legend(loc='lower left')
ax.set_ylim(17, 26)

# --- Right panel: Distribution of FTE by group and period ---
ax2 = axes[1]
colors = {'NJ': 'steelblue', 'PA': 'darkorange'}
for state in ['NJ', 'PA']:
    for period in ['pre', 'post']:
        subset = df[(df['state'] == state) & (df['period'] == period)]['fte']
        linestyle = '-' if period == 'post' else '--'
        alpha = 0.8 if period == 'post' else 0.4
        ax2.hist(subset, bins=25, alpha=alpha, color=colors[state],
                 histtype='step', linewidth=2, linestyle=linestyle,
                 label=f'{state} {period}')

ax2.set_xlabel('FTE Employment')
ax2.set_ylabel('Count')
ax2.set_title('FTE Distribution by Group and Period')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../resources/did_card_krueger.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Counterfactual NJ post: {nj_counterfactual:.2f} FTE")
print(f"Observed NJ post:       {nj_post_mean:.2f} FTE")
print(f"Treatment effect:        {did_estimate:+.2f} FTE")

## 4. Regression DiD

The OLS regression formulation estimates the same quantity:

$$\text{fte}_{it} = \alpha + \beta \cdot \text{post}_t + \gamma \cdot \text{treated}_i + \tau \cdot (\text{post}_t \times \text{treated}_i) + \epsilon_{it}$$

We add restaurant chain fixed effects to improve precision.

In [ ]:
# Basic DiD regression (no covariates)
model_basic = smf.ols('fte ~ post + treated + post:treated', data=df).fit(cov_type='HC1')

# DiD with chain fixed effects
model_fe = smf.ols('fte ~ post + treated + post:treated + C(chain)', data=df).fit(cov_type='HC1')

# Print comparison table
print("DiD Estimates".center(55))
print("=" * 55)
print(f"{'Model':<30} {'Estimate':>8} {'SE':>8} {'95% CI':>14}")
print("-" * 55)
for name, model in [("Basic DiD", model_basic), ("DiD + Chain FE", model_fe)]:
    coef = model.params['post:treated']
    se = model.bse['post:treated']
    ci_lo, ci_hi = model.conf_int().loc['post:treated']
    print(f"{name:<30} {coef:>8.3f} {se:>8.3f} [{ci_lo:>5.2f}, {ci_hi:>5.2f}]")
print("-" * 55)
print(f"{'True value (Card & Krueger)':<30} {TRUE_DiD:>8.3f}")

print("\n--- Full Regression Output: Basic DiD ---")
print(model_basic.summary().tables[1])

## 5. Bayesian DiD with CausalPy

CausalPy's `DifferenceInDifferences` class uses PyMC to estimate the full posterior distribution of the treatment effect. This gives us richer uncertainty quantification than standard confidence intervals.

In [ ]:
# Fit Bayesian DiD using CausalPy
bayes_did = cp.DifferenceInDifferences(
    data=df,
    formula='fte ~ 1 + post + treated + post:treated',
    time_variable_name='post',
    group_variable_name='treated',
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            'draws': 2000,
            'tune': 1000,
            'chains': 4,
            'target_accept': 0.9,
            'progressbar': True
        }
    )
)

print("Model fitted. Checking convergence...")

In [ ]:
# Convergence diagnostics
summary = az.summary(bayes_did.idata)
print("Convergence Diagnostics (R̂ < 1.01 = good):")
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']].round(3))

In [ ]:
# Extract treatment effect posterior
tau_samples = bayes_did.idata.posterior['post:treated'].values.flatten()

print("=" * 45)
print("BAYESIAN DiD RESULTS")
print("=" * 45)
print(f"Posterior mean:    {tau_samples.mean():.3f} FTE")
print(f"Posterior std:     {tau_samples.std():.3f} FTE")
print(f"94% HDI:           [{np.percentile(tau_samples, 3):.2f}, {np.percentile(tau_samples, 97):.2f}]")
print(f"P(τ > 0):          {(tau_samples > 0).mean():.3f}")
print(f"P(τ > 1 FTE):      {(tau_samples > 1).mean():.3f}")
print(f"True value:        {TRUE_DiD:.3f} FTE")

In [ ]:
# Full visualisation: CausalPy plot + posterior histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CausalPy's built-in DiD plot
bayes_did.plot(ax=axes[0])
axes[0].set_title('CausalPy DiD: Employment Effect of NJ Minimum Wage')
axes[0].set_ylabel('Mean FTE Employment')

# Posterior distribution
axes[1].hist(tau_samples, bins=60, color='steelblue', edgecolor='white',
             alpha=0.8, density=True)
axes[1].axvline(0, color='red', ls='--', linewidth=2, label='No effect (τ=0)')
axes[1].axvline(tau_samples.mean(), color='black', ls='-', linewidth=2,
                label=f'Posterior mean = {tau_samples.mean():.2f}')
axes[1].axvline(TRUE_DiD, color='green', ls=':', linewidth=2,
                label=f'True value = {TRUE_DiD:.2f}')

# HDI shading
hdi_lo, hdi_hi = np.percentile(tau_samples, [3, 97])
x_range = np.linspace(tau_samples.min(), tau_samples.max(), 300)
axes[1].fill_betweenx([0, 0.08], hdi_lo, hdi_hi,
                       alpha=0.15, color='steelblue', label=f'94% HDI [{hdi_lo:.2f}, {hdi_hi:.2f}]')

axes[1].set_xlabel('Treatment Effect (FTE Employees)')
axes[1].set_ylabel('Posterior Density')
axes[1].set_title('Posterior Distribution of DiD Treatment Effect')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. Assessing Parallel Trends

With only two periods (pre/post), we cannot formally test parallel trends. But we can examine whether the pre-period levels and industry composition are similar between NJ and PA, and whether there are any observable time-varying confounders.

In the original Card & Krueger study, the authors argued that:
1. NJ and PA fast food industries were structurally similar before the change
2. PA had no concurrent wage legislation
3. Chain distribution was similar between states
4. A later study with quarterly data showed flat pre-trends in the few quarters before the increase

In [ ]:
# Check pre-period comparability
pre_data = df[df['period'] == 'pre']

print("Pre-Period Comparability Check")
print("=" * 45)
print(f"\nMean FTE (pre):")
print(pre_data.groupby('state')['fte'].agg(['mean', 'std', 'count']).round(2))

print(f"\nChain distribution:")
chain_dist = pd.crosstab(pre_data['state'], pre_data['chain'], normalize='index').round(3)
print(chain_dist)

# T-test on pre-period means
from scipy.stats import ttest_ind
nj_pre_fte = pre_data[pre_data['state'] == 'NJ']['fte']
pa_pre_fte = pre_data[pre_data['state'] == 'PA']['fte']
t_stat, p_val = ttest_ind(nj_pre_fte, pa_pre_fte)
print(f"\nPre-period t-test: t={t_stat:.2f}, p={p_val:.3f}")
print("(NJ and PA have significantly different pre-period means —")
print(" but parallel TRENDS, not equal LEVELS, is what's required)")

## 7. Interpretation

The DiD estimate says:

> New Jersey fast food restaurants increased full-time equivalent employment by approximately **2.7 FTE** after the minimum wage increase, relative to what would have been expected based on Pennsylvania's trend.

This is an estimate of the **Average Treatment Effect on the Treated (ATT)** — the effect for NJ restaurants specifically.

**Why might employment rise with a minimum wage increase?**

Card & Krueger proposed several mechanisms:
1. **Monopsony power:** Fast food employers had wage-setting power below competitive level; minimum wage pushed towards the efficient level
2. **Efficiency wages:** Higher wages reduce turnover, improving productivity
3. **Demand effects:** Low-wage workers have high marginal propensity to consume; higher wages boost local demand

The result was controversial and sparked a large follow-on literature. The identification hinges on the parallel trends assumption — NJ and PA would have had the same employment trend absent the minimum wage increase.

In [ ]:
# Summary of all estimates
print("FINAL COMPARISON: DiD ESTIMATES")
print("=" * 55)
ols_coef = model_basic.params['post:treated']
ols_ci = model_basic.conf_int().loc['post:treated'].values
ols_fe_coef = model_fe.params['post:treated']
ols_fe_ci = model_fe.conf_int().loc['post:treated'].values
bayes_mean = tau_samples.mean()
bayes_hdi = np.percentile(tau_samples, [3, 97])

print(f"{'Method':<25} {'Estimate':>8} {'Lower':>8} {'Upper':>8}")
print("-" * 55)
print(f"{'OLS (basic)':<25} {ols_coef:>8.2f} {ols_ci[0]:>8.2f} {ols_ci[1]:>8.2f}")
print(f"{'OLS + Chain FE':<25} {ols_fe_coef:>8.2f} {ols_fe_ci[0]:>8.2f} {ols_fe_ci[1]:>8.2f}")
print(f"{'Bayesian DiD':<25} {bayes_mean:>8.2f} {bayes_hdi[0]:>8.2f} {bayes_hdi[1]:>8.2f}")
print("-" * 55)
print(f"{'True value':<25} {TRUE_DiD:>8.2f}")
print("\nAll three methods recover the true effect within their uncertainty ranges.")

## Self-Check

1. Change `TRUE_DiD` by setting NJ_POST_MEAN to 20.0 (negative effect). Do all three estimators correctly detect a negative treatment effect?

2. What happens to the DiD estimate if you reduce the sample size to 50 restaurants per group? Run the code with `n_nj = 50, n_pa = 50` and observe how the confidence/credible intervals change.

3. Add a continuous covariate — say, `years_open` — that is correlated with FTE but balanced across NJ and PA. Does adding it to the regression change the DiD estimate? Why or why not?

4. Simulate a violation of parallel trends: make PA employment rise faster than NJ in the post-period due to an economic boom, independent of any minimum wage effect. How does this bias the DiD estimate?

---
**Next:** `02_staggered_did.ipynb` — Staggered DiD with multiple treatment cohorts

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])